# Running apps-benchmark from a notebook

This demo shows the interactive `Benchmarker` orchestration layer for cataloging benchmark cases, inspecting algorithms, running a small local benchmark, and using async submission records.

In [ ]:
from apps_benchmark import Benchmarker
from apps_benchmark.backends.mock_backend import MockBackend

benchmarker = Benchmarker(include_diy=False)
cases = benchmarker.get_benchmark_cases()
cases.head()

In [ ]:
algorithms = benchmarker.get_algorithms()
algorithms.head(10)

In [ ]:
qft_cases = benchmarker.get_benchmark_cases(
    benchmark_category="qft",
    max_qubits=10,
    runnable_only=True,
)
qft_cases

In [ ]:
benchmarker.get_benchmark_circuit_stats(
    qft_cases,
    solution_algorithms="cosine_qft",
)

In [ ]:
sync_results = benchmarker.run_benchmarks(
    qft_cases,
    backend=MockBackend(),
    shots=[128, 512],
    solution_algorithms="cosine_qft",
)
sync_results[["instance_name", "solution_algorithm", "backend", "shots_per_qc", "score", "status"]]

## Async Mode

`submit_benchmarks` works with backends that implement `AbstractAsyncBackend`. The tiny backend below completes immediately, which makes the submit/load/update record flow runnable without external credentials.

In [ ]:
from pathlib import Path

import pandas as pd
from qiskit import QuantumCircuit

from apps_benchmark.core.backend import AbstractAsyncBackend, JobData, JobStatus, MeasurementBatch


class InstantAsyncBackend(AbstractAsyncBackend):
    def __init__(self, status: JobStatus = JobStatus.DONE) -> None:
        self.status = status
        self.jobs: dict[str, JobData] = {}

    def name(self) -> str:
        return "instant_async"

    def submit(
        self,
        circuits: list[QuantumCircuit],
        shots: int = 1000,
        job_name: str | None = None,
    ) -> tuple[str, JobData]:
        job_id = f"instant_job_{len(self.jobs)}"
        job_data = self.serialize_job_data(circuits, shots, job_name or "")
        job_data["num_qubits"] = [circuit.num_qubits for circuit in circuits]
        self.jobs[job_id] = job_data
        return job_id, job_data

    def job_status(self, job_id: str) -> JobStatus:
        return self.status

    def retrieve_results(
        self,
        job_id: str,
        job_data: JobData,
    ) -> tuple[MeasurementBatch, pd.Timestamp]:
        shots = int(job_data["shots"])
        num_qubits = [int(value) for value in job_data["num_qubits"]]
        measurements = [{"0" * nqubits: shots} for nqubits in num_qubits]
        return measurements, pd.Timestamp.now(tz="UTC")

In [ ]:
record_db = Path("qft_async_demo_record.json")
async_backend = InstantAsyncBackend()

submission = benchmarker.submit_benchmarks(
    qft_cases,
    backend=async_backend,
    shots=128,
    record_db=record_db,
    solution_algorithms="cosine_qft",
)
submission[["instance_name", "solution_algorithm", "backend", "shots_per_qc", "status", "job_id"]]

In [ ]:
loaded = benchmarker.load_submission_record(record_db)
updated = benchmarker.update_submission_record(record_db, backend=async_backend)

updated[["instance_name", "status", "score", "time_to_soln", "adjusted_tts"]]